# Exploratory Data Analysis — Legal Datasets

This notebook downloads legal-domain HuggingFace datasets and provides:
- Split sizes (train / test / val)
- Column types and schema
- Category / metadata distributions with plots
- Example records for manual inspection

In [ ]:
from collections import Counter

import datasets
import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset

## Helper functions

In [ ]:
def summarise_dataset(ds_dict, name="Dataset"):
    """Print split sizes and column info for a DatasetDict."""
    print(f"=== {name} ===")
    print(f"Splits: {list(ds_dict.keys())}")
    print()
    for split, ds in ds_dict.items():
        print(f"  {split}: {len(ds):,} rows, {len(ds.column_names)} columns")
    print()
    # Show schema from first available split
    first = next(iter(ds_dict.values()))
    print("Columns:")
    for col, feat in first.features.items():
        print(f"  {col}: {feat}")
    print()


def plot_split_sizes(ds_dict, title="Split sizes"):
    """Bar chart of number of rows per split."""
    splits = list(ds_dict.keys())
    sizes = [len(ds_dict[s]) for s in splits]
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(splits, sizes)
    ax.set_ylabel("Number of rows")
    ax.set_title(title)
    for i, v in enumerate(sizes):
        ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.show()


def plot_categorical(ds, column, title=None, top_n=20):
    """Plot value counts for a categorical column."""
    counts = Counter(ds[column])
    most_common = counts.most_common(top_n)
    labels, values = zip(*most_common, strict=False)
    fig, ax = plt.subplots(figsize=(8, max(3, len(labels) * 0.35)))
    ax.barh(range(len(labels)), values)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel("Count")
    ax.set_title(title or f"Distribution of '{column}' (top {top_n})")
    plt.tight_layout()
    plt.show()


def plot_text_lengths(ds, column, title=None, bins=50):
    """Histogram of string lengths for a text column."""
    lengths = [len(str(x)) for x in ds[column]]
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(lengths, bins=bins, edgecolor="black", linewidth=0.5)
    ax.set_xlabel("Character length")
    ax.set_ylabel("Count")
    ax.set_title(title or f"Text length distribution: '{column}'")
    plt.tight_layout()
    plt.show()
    sr = pd.Series(lengths)
    print(sr.describe().to_string())


def show_example(ds, idx=0):
    """Pretty-print a single example for manual inspection."""
    example = ds[idx]
    for key, value in example.items():
        display_val = str(value)
        if len(display_val) > 500:
            display_val = display_val[:500] + "..."
        print(f"--- {key} ---")
        print(display_val)
        print()

---
## Dataset 1: LEXam-Benchmark/LEXam

Legal examination benchmark dataset.

In [ ]:
lexam = load_dataset("LEXam-Benchmark/LEXam", "open_question")
# summarise_dataset(lexam, name="LEXam")

# Filter lexam test to only have english examples
lexam_test_en = lexam["test"].filter(lambda x: x["language"] == "en")

In [ ]:
plot_split_sizes(lexam, title="LEXam — split sizes")

In [ ]:
# Identify categorical vs text columns and plot accordingly
first_split = next(iter(lexam.values()))
string_cols = [
    c
    for c, f in first_split.features.items()
    if str(f).startswith("Value") or "string" in str(f).lower()
]
print("String columns:", string_cols)

for col in first_split.column_names:
    unique = len(set(first_split[col][:5000]))  # sample for speed
    total = min(5000, len(first_split))
    ratio = unique / total if total > 0 else 1
    if 1 < unique <= 50:
        print(f"\nPlotting categorical column: {col} ({unique} unique values)")
        plot_categorical(first_split, col, title=f"LEXam — {col}")

In [ ]:
# Text length distribution for the first long-text column
for col in first_split.column_names:
    sample_val = str(first_split[0][col])
    if len(sample_val) > 100:
        print(f"Text length distribution for: {col}")
        plot_text_lengths(first_split, col, title=f"LEXam — {col} lengths")
        break

### LEXam — English open questions

Filter for `language == "en"` on the `open_question` config and inspect metadata distributions.

In [ ]:
lexam_open = load_dataset("LEXam-Benchmark/LEXam", "open_question")
# Combine all splits, then filter to English
all_open = datasets.concatenate_datasets(list(lexam_open.values()))
en_open = all_open.filter(lambda x: x["language"] == "en")
print(f"Open-question rows (all languages): {len(all_open):,}")
print(f"Open-question rows (English only):  {len(en_open):,}")
print(f"Columns: {en_open.column_names}")

In [ ]:
metadata_cols = ["course", "area", "jurisdiction", "year"]
for col in metadata_cols:
    plot_categorical(en_open, col, title=f"LEXam English open questions — {col}")

In [ ]:
# Question and answer length distributions
plot_text_lengths(en_open, "question", title="LEXam English open questions — question lengths")
print()
plot_text_lengths(en_open, "answer", title="LEXam English open questions — answer lengths")

In [ ]:
print("=== LEXam English open question — Example record ===")
show_example(en_open, idx=0)

### Interactive example viewer — sorted by question length

In [ ]:
import html as _html

import ipywidgets as widgets
from IPython.display import HTML, display

# --- Data preparation ---
df_viewer = en_open.to_pandas()
df_viewer["q_len"] = df_viewer["question"].str.len()
df_viewer["a_len"] = df_viewer["answer"].str.len()
df_viewer = df_viewer.sort_values(["q_len", "a_len"]).reset_index(drop=True)
total = len(df_viewer)

# --- Widgets ---
btn_prev = widgets.Button(description="\u25c0 Prev", layout=widgets.Layout(width="100px"))
btn_next = widgets.Button(description="Next \u25b6", layout=widgets.Layout(width="100px"))
counter = widgets.HTML(value="")
out = widgets.Output()

state = [0]


def render(idx):
    row = df_viewer.iloc[idx]
    counter.value = (
        f'<span style="font-family:monospace; font-size:14px;">Example {idx + 1} / {total}</span>'
    )
    q = _html.escape(str(row["question"]))
    a = _html.escape(str(row["answer"]))
    card = f"""
    <div style="font-family: system-ui, sans-serif; max-width: 900px;">
      <div style="margin-bottom: 8px; color: #555; font-size: 13px;">
        <span style="background:#e8eaf6; padding:2px 8px; border-radius:10px; margin-right:6px;">
          {_html.escape(str(row["course"]))}</span>
        <span style="background:#e0f2f1; padding:2px 8px; border-radius:10px; margin-right:6px;">
          {_html.escape(str(row["area"]))}</span>
        <span style="background:#fff3e0; padding:2px 8px; border-radius:10px; margin-right:6px;">
          {_html.escape(str(row["jurisdiction"]))}</span>
        <span style="background:#fce4ec; padding:2px 8px; border-radius:10px;">
          {_html.escape(str(row["year"]))}</span>
      </div>
      <div style="border-left:4px solid #1565c0; background:#f5f7ff;
                  padding:12px 16px; margin-bottom:12px; border-radius:0 4px 4px 0;">
        <div style="font-weight:600; color:#1565c0; margin-bottom:6px; font-size:14px;">
          Question ({row["q_len"]} chars)</div>
        <div style="white-space:pre-wrap; font-size:14px; line-height:1.5;">{q}</div>
      </div>
      <div style="border-left:4px solid #2e7d32; background:#f5fdf5;
                  padding:12px 16px; border-radius:0 4px 4px 0;">
        <div style="font-weight:600; color:#2e7d32; margin-bottom:6px; font-size:14px;">
          Answer ({row["a_len"]} chars)</div>
        <div style="white-space:pre-wrap; font-size:14px; line-height:1.5;">{a}</div>
      </div>
    </div>
    """
    out.clear_output(wait=True)
    with out:
        display(HTML(card))


def on_prev(b):
    if state[0] > 0:
        state[0] -= 1
        render(state[0])


def on_next(b):
    if state[0] < total - 1:
        state[0] += 1
        render(state[0])


btn_prev.on_click(on_prev)
btn_next.on_click(on_next)

nav_bar = widgets.HBox(
    [btn_prev, counter, btn_next],
    layout=widgets.Layout(align_items="center", gap="12px", margin="0 0 8px 0"),
)
viewer = widgets.VBox([nav_bar, out])

render(0)
display(viewer)

In [ ]:
print("=== LEXam — Example record ===")
show_example(first_split, idx=0)

---
## Dataset 2: isaacus/legal-rag-bench

Legal RAG benchmark dataset.

In [ ]:
legal_rag = load_dataset("isaacus/legal-rag-bench")
summarise_dataset(legal_rag, name="legal-rag-bench")

In [ ]:
plot_split_sizes(legal_rag, title="legal-rag-bench — split sizes")

In [ ]:
first_split_rag = next(iter(legal_rag.values()))

for col in first_split_rag.column_names:
    unique = len(set(first_split_rag[col][:5000]))
    if 1 < unique <= 50:
        print(f"\nPlotting categorical column: {col} ({unique} unique values)")
        plot_categorical(first_split_rag, col, title=f"legal-rag-bench — {col}")

In [ ]:
for col in first_split_rag.column_names:
    sample_val = str(first_split_rag[0][col])
    if len(sample_val) > 100:
        print(f"Text length distribution for: {col}")
        plot_text_lengths(first_split_rag, col, title=f"legal-rag-bench — {col} lengths")
        break

In [ ]:
print("=== legal-rag-bench — Example record ===")
show_example(first_split_rag, idx=0)

---
## Loading a different dataset

To explore another HuggingFace dataset, change the identifier below and re-run:

In [ ]:
# DATASET_ID = "your-org/your-dataset"
# ds = load_dataset(DATASET_ID)
# summarise_dataset(ds, name=DATASET_ID)
# plot_split_sizes(ds)
# first = next(iter(ds.values()))
# show_example(first, idx=0)